# 06 · Can a buyer rerun this tomorrow?

Batch scoring combines the saved forecast model with the store's latest completed sales and price history, a known calendar, on-hand stock, and firm receipts. That last piece matters. A forecast for 20 units has a different order suggestion for a store with 2 usable units than for one with 18.

This example uses real M5 recorded sales up to April 24, 2016 and a synthetic inventory and inbound snapshot. The order file is a demonstration. It contains recommendations for review and sends no orders to a retailer.

In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
root = next((place for place in (current, *current.parents) if (place / "configs/config.yaml").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_audit import load_config

config_path = root / "configs/config.yaml"
config = load_config(config_path)
forecast = root / config["forecast"]["output_dir"]
report = root / config["artifacts"]["report_dir"] / "inventory"
processed = root / config["data"]["processed_dir"]

from src.inference import monitor_from_files, recommend_order_table, snapshot
from src.models import load_bundle

if not (report / "policy.json").is_file():
    raise FileNotFoundError("Run Notebook 05 or the inventory CLI before scoring new orders.")
store_id = config["data"]["stores"][0]
origin = "2016-04-24"
lead = max(config["inventory"]["lead_times"])
bundle = forecast / "bundle"
stock_file = root / "examples/inventory_snapshot.csv"
inbound_file = root / "examples/inbound_snapshot.csv"
policy_file = report / "policy.json"
metadata = load_bundle(bundle)
display(pd.DataFrame([
    {"kind": "forecast bundle", "value": metadata["selection"]["cumulative"]["selected"]},
    {"kind": "review date", "value": str((pd.Timestamp(origin) + pd.Timedelta(days=1)).date())},
    {"kind": "lead days", "value": str(lead)},
]))
inputs = {
    "history": processed / "panel" / f"{store_id}.parquet",
    "prices": processed / "price_history" / f"{store_id}.parquet",
    "calendar": processed / "calendar.parquet",
    "bundle": bundle, "store": store_id,
}

## 1. Reject a stale or ambiguous snapshot

Stock is counted at close of the last completed sales day. A shipment due the next morning is added to usable stock before making the decision. Later firm shipments already on order count in inventory position and are shown separately. In this fixed-lead-time setting an overdue receipt or one beyond the oldest possible due date suggests the data are inconsistent with the assumed pipeline, so scoring stops.

For a new operating day, provide the previous full daily history or append the next completed sales rows with the batch command. The scorer rejects missing days, missing SKUs, duplicate inventory counts, and unknown inbound items. Future calendar dates may be used as scheduled covariates; selling prices need completed weeks.

In [ ]:
orders = recommend_order_table(
    config_path, **inputs, inventory=stock_file, inbound=inbound_file,
    policy=policy_file, as_of=origin, lead=lead,
)
wanted = ["item_id", "regime", "model", "rule", "target_units", "on_hand", "due_on_review",
          "outstanding_after_review_receipts", "inventory_position", "recommended_units", "order_due_date"]
display(orders.loc[orders["due_on_review"].gt(0), wanted].head(6).round({"target_units": 2}))
assert len(orders) == config["data"]["expected_series_per_store"]
assert (orders["price_week_end"].dropna() <= pd.Timestamp(origin)).all()
assert orders["origin"].eq(pd.Timestamp(origin)).all()
assert orders["order_due_date"].eq(pd.Timestamp(origin) + pd.Timedelta(days=lead + 1)).all()
np.testing.assert_array_equal(
    orders["recommended_units"], np.ceil(np.maximum(0, orders["target_units"] - orders["inventory_position"]))
)
print(f"PASS: {len(orders):,} suggestions, {(orders['recommended_units'] > 0).sum():,} positive order lines.")

## 2. Learn what changed before reading service

Recent sales may shift even if the model file is identical. Compare the latest 28 days with training across regimes, and report how many items lack an eligible price. A change in the zero share or average sales can reflect seasonality, availability or assortment. It tells a buyer where to investigate and does not by itself identify why the forecast changed.

For actual service reporting, feed observed service rows with demand, fulfilled units and lost units from an inventory system. The unit-fill rate applies to the supplied ledger rows. The three-row example below confirms the service arithmetic, and it supplies no evidence about actual Walmart service.

In [ ]:
monitoring = monitor_from_files(
    config_path, **inputs, orders=orders, ledger=root / "examples/service_ledger.csv",
)
display(pd.DataFrame(monitoring["drift"]).round(4))
display(pd.DataFrame([monitoring["service"]]))
assert monitoring["service"]["fill_rate"] == 7 / 8

### What if the inventory system sent yesterday's stock?

A fresh forecast cannot rescue a stale stock position. Change the date on one row and confirm the scorer rejects it before calculating order quantities.

In [ ]:
stock = pd.read_csv(stock_file)
stock.loc[0, "as_of"] = "2016-04-23"
try:
    snapshot(stock, pd.read_csv(inbound_file), orders["item_id"].to_numpy(), store_id, origin, lead)
except ValueError as error:
    assert "as-of" in str(error)
    print(f"PASS: {error}")
else:
    raise AssertionError("Stale stock passed validation.")

## 3. Use the same path outside Jupyter

The [deployment instructions](../docs/BATCH_DEPLOYMENT.md) show the full command, the input columns and Docker mounts. The output records the forecast origin, planned due date, demand quantiles, stock position, order units, model choice, eligible price week, and model and policy fingerprints. Keeping these together makes an order recommendation possible to investigate later.

The April 25 review is after the last CA_1 recorded sales date in the supplied file. This example shows the model can score the decision from available history and known schedule. Later realized demand is needed to measure its forecast error or the resulting service.